In [1]:
%run utils.py

In [2]:
import os
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.12:3.5.7 pyspark-shell'

spark = get_spark_session(app_name="Streaming")

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-avro_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0fbde5be-8353-4d1b-92b5-4a3185ff49b3;1.0
	confs: [default]
	found org.apache.spark#spark-avro_2.12;3.5.7 in central
	found org.tukaani#xz;1.9 in central
downloading https://repo1.maven.org/maven2/org/apache/spark/spark-avro_2.12/3.5.7/spark-avro_2.12-3.5.7.jar ...
	[SUCCESSFUL ] org.apache.spark#spark-avro_2.12;3.5.7!spark-avro_2.12.jar (407ms)
downloading https://repo1.maven.org/maven2/org/tukaani/xz/1.9/xz-1.9.jar ...
	[SUCCESSFUL ] org.tukaani#xz;1.9!xz.jar (131ms)
:: resolution report :: resolve 2671ms :: artifacts dl 543ms
	:: modules in use:
	org.apache.spark#spark-avro_2.12;3.5.7 from central in [default]
	org.tukaani#xz;1.9 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules    

SparkSession started with app name: Streaming


In [3]:
# ─────────────────────────────────────────────
# APICURIO SCHEMA REGISTRY HELPER
# Apicurio REST API is similar to Confluent but 
# uses /apis/ccompat/v7 for Confluent-compat endpoint
# OR /apis/registry/v2 for native Apicurio endpoint
# ─────────────────────────────────────────────
import requests
import json

def get_envelope_schema(schema_registry_url, subject):
    url = f"{schema_registry_url}/apis/ccompat/v7/subjects/{subject}/versions/latest"
    response = requests.get(url)
    return response.json()["schema"]

def get_table_fields(schema_registry_url, subject):
    envelope_schema = json.loads(get_envelope_schema(schema_registry_url, subject))
    before_field = next(f for f in envelope_schema["fields"] if f["name"] == "before")
    value_schema = next(t for t in before_field["type"] if isinstance(t, dict))
    return [f["name"] for f in value_schema["fields"]]

def check_bootstrap(query):
    if not query.isActive:
        print("Stream is STOPPED")                                                                                                                
        return                                                                                                                                    
                                                                                                                                                
    p = query.lastProgress                                                                                                                        
    if not p:                                                                                                                                     
        print("Waiting for first batch...")                                                                                                       
        return                                                                                                                                    
                                                                                                                                                
    num_input = p["numInputRows"]                                                                                                                 
    status = "STILL CATCHING UP" if num_input >= 1 else "CAUGHT UP — safe to switch to steady-state"                                           
    print(f"batch {p['batchId']}: {num_input:,} rows  ← {status}")
    # When you see "CAUGHT UP", stop and restart: 
    return num_input

## 1. REUSABLE BRONZE STREAMING FUNCTION

In [7]:
# ─────────────────────────────────────────────
# REUSABLE BRONZE STREAMING FUNCTION
# ─────────────────────────────────────────────
from pyspark.sql.functions import col, when, to_date, concat, lit, sha2
from pyspark.sql.avro.functions import from_avro

KAFKA_BROKER = "host.docker.internal:9092"
APICURIO_URL = "http://host.docker.internal:18081"
BRONZE_DB    = "bronze"

def start_bronze_stream(topic, bronze_table, nature_key=None, startingOffsets="earliest", max_offsets_per_trigger=None, process_time="3 seconds"):
    """Start a CDC streaming query from Kafka to Hudi bronze table."""
    bronze_table = bronze_table.lower()
    
    envelope_schema = get_envelope_schema(APICURIO_URL, f"{topic}-value")
    table_fields    = get_table_fields(APICURIO_URL, f"{topic}-value")

    raw_df = (
        spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BROKER)
        .option("subscribe", topic)
        .option("startingOffsets", startingOffsets)
        .option("maxOffsetsPerTrigger", max_offsets_per_trigger) 
        .load()
    )

    deserialized_df = (
        raw_df
        .filter(col("value").isNotNull())
        .withColumnRenamed("value", "avro_payload")
        .select(
            from_avro(col("avro_payload"), envelope_schema, {"mode": "PERMISSIVE"}).alias("envelope"),
            col("offset").alias("_kafka_offset"),
            col("partition").alias("_kafka_partition"),
            col("timestamp").alias("_kafka_ingest_ts"),
        )
    )
    
    # Flatten: data columns + CDC metadata ────────────
    # ┌──────────────┬─────────────┬────────────────────────────────┬─────────────────────────────┐
    # │    CDC op    │ envelope.op │           Takes from           │ Handles NULL column values? │
    # ├──────────────┼─────────────┼────────────────────────────────┼─────────────────────────────┤
    # │ snapshot (r) │ "r"         │ after                          │ Yes                         │
    # ├──────────────┼─────────────┼────────────────────────────────┼─────────────────────────────┤
    # │ create (c)   │ "c"         │ after                          │ Yes                         │    
    # ├──────────────┼─────────────┼────────────────────────────────┼─────────────────────────────┤
    # │ update (u)   │ "u"         │ after (even if column is NULL) │ Yes                         │    
    # ├──────────────┼─────────────┼────────────────────────────────┼─────────────────────────────┤
    # │ delete (d)   │ "d"         │ before                         │ Yes                         │
    # └──────────────┴─────────────┴────────────────────────────────┴─────────────────────────────┘
    data_cols = [
        when(col("envelope.op") == "d", col(f"envelope.before.{f}"))
        .otherwise(col(f"envelope.after.{f}"))
        .alias(f)
        for f in table_fields
    ]

    meta_cols = [
        col("envelope.op").alias("_cdc_op"),
        col("envelope.ts_ms").alias("_cdc_ts_ms"),
        col("envelope.source.ts_ms").alias("_source_ts_ms"),
        col("_kafka_offset"),
        col("_kafka_partition"),
        col("_kafka_ingest_ts"),
    ]

    key_expr = (
        col(nature_key)
        if nature_key
        else sha2(concat(
            col("_kafka_partition").cast("string"), lit(":"),
            col("_kafka_offset").cast("string")
        ), 256)
    )

    bronze_df = (
        deserialized_df
        .filter(col("envelope").isNotNull())
        .select(*data_cols, *meta_cols)
        .withColumn("__deleted", when(col("_cdc_op") == "d", True).otherwise(False))
        .withColumn("_ingest_date", to_date(col("_kafka_ingest_ts")))
        .withColumn("_record_key", key_expr)
    )

    base_path  = f"s3a://{BRONZE_DB}/{bronze_table}"
    checkpoint = f"s3a://{BRONZE_DB}/{bronze_table}/checkpoints"

    write_operation = "upsert" if nature_key else "insert"
    
    hudi_conf = {
        "hoodie.table.name": bronze_table,
        "hoodie.database.name": BRONZE_DB,
        "hoodie.datasource.write.recordkey.field": "_record_key",
        "hoodie.datasource.write.table.type": "COPY_ON_WRITE",
        "hoodie.datasource.write.operation": write_operation,
        "hoodie.datasource.write.partitionpath.field": "_ingest_date",
        "hoodie.datasource.write.hive_style_partitioning": "true",
        "hoodie.datasource.write.precombine.field": "_kafka_offset",
        "hoodie.datasource.hive_sync.mode": "hms",
        "hoodie.datasource.hive_sync.jdbcurl": "thrift://hive-metastore:9083",
        "hoodie.datasource.hive_sync.enable": "true",
        "hoodie.datasource.hive_sync.support_timestamp": "true",
        "hoodie.upsert.shuffle.parallelism": 2,
        "hoodie.insert.shuffle.parallelism": 2,
        "hoodie.parquet.compression.codec": "snappy",
    }

    return (
        bronze_df.writeStream
        .format("hudi")
        .options(**hudi_conf)
        .outputMode("append")
        .option("path", base_path)
        .option("checkpointLocation", checkpoint)
        .trigger(processingTime=process_time)
        .start()
    )

## 2. public.orders -> bronze.orders

In [8]:
streaming_orders = start_bronze_stream(
    topic="cdc_postgres.public.orders", 
    bronze_table="postgres_public_orders", 
    nature_key="id"
)

# WARNING: Unable to get Instrumentation. Dynamic Attach failed. You may add this JAR as -javaagent manually, or supply -Djdk.attach.allowAttachSelf


[Stage 3:>                                                          (0 + 1) / 1]

# WARNING: Unable to attach Serviceability Agent. sun.jvm.hotspot.memory.Universe.getNarrowOopBase()


## 3. CRBT.sub_collection_log -> bronze.pg_crbt_sub_collection_logs

In [ ]:
streaming_sub_collection_log = start_bronze_stream(
    topic="cdc_postgres.CRBT.sub_collection_log", 
    bronze_table="pg_crbt_sub_collection_logs", 
    nature_key=None
)

## 4. CRBT.substate_log -> bronze.pg_crbt_substate_log

In [5]:
# Phase 1: Bootstrap
BOOTSTRAP_BATCH = 500000
bootsrap_streaming_substate_log = start_bronze_stream(
    topic="cdc_postgres.CRBT.substate_log", 
    bronze_table="pg_crbt_substate_log", 
    nature_key=None,
    max_offsets_per_trigger=BOOTSTRAP_BATCH
)                                                                                                                                    

In [26]:
check_bootstrap(bootsrap_streaming_substate_log)

batch 10: 0 rows  ← CAUGHT UP — safe to switch to steady-state


0

In [27]:
# When you see "CAUGHT UP", stop and restart:  
bootsrap_streaming_substate_log.stop()

In [28]:
# Phase 2: Steady-state stream
streaming_substate_log = start_bronze_stream(
    topic="cdc_postgres.CRBT.substate_log", 
    bronze_table="pg_crbt_substate_log", 
    nature_key=None
)

## 5. CRBT.charge_log -> bronze.pg_crbt_charge_log

In [29]:
# Phase 1: Bootstrap
BOOTSTRAP_BATCH = 1000000
bootsrap_streaming_charge_log = start_bronze_stream(
    topic="cdc_postgres.CRBT.charge_log", 
    bronze_table="pg_crbt_charge_log", 
    nature_key=None,
    max_offsets_per_trigger=BOOTSTRAP_BATCH
)

[Stage 217:>                                                        (0 + 1) / 1]

In [41]:
check_bootstrap(bootsrap_streaming_charge_log)

batch 30: 0 rows  ← CAUGHT UP — safe to switch to steady-state


0